## Reads the PID log messages and builds a dictionary with the results

Craig Lage
26-Mar-26

In [ ]:
# Times Square Parameters
day_obs = 20260325
store_dir = "/home/c/cslage/u/MTAOS/times_square_reports/" # Where to store the results

In [ ]:
import numpy as np
import pickle as pkl
import re
import matplotlib.pyplot as plt
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData, getMostRecentRowWithDataBefore
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId

In [ ]:
client = makeEfdClient()
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=True)

In [ ]:
exposureList = []
for record in butler.registry.queryDimensionRecords("exposure", 
                    where=f"exposure.day_obs={dayObs} and instrument='LSSTCam'"):
    exposureList.append([record.id, record])
exposureList.sort(key=lambda x: x[0])

pidDict = {}
for exposure in exposureList:
    [expId, record] = exposure
    pidData = getEfdData(client, 
                         topic='lsst.sal.MTAOS.logevent_logMessage',
                         columns=['message'],
                         expRecord=record)
    dof_event = getMostRecentRowWithDataBefore(
        client,
        "lsst.sal.MTAOS.logevent_degreeOfFreedom",
        timeToLookBefore=record.timespan.end,
    )
    if len(pidData) == 0 or len(dof_event) == 0:
        continue
    for msg in pidData['message'].values:
        if 'PID' in msg:
            matches = re.findall(r'\[([^\]]+)\]', msg, re.DOTALL)
            arrays = [np.fromstring(m, sep=' ') for m in matches]
            kp_array = np.array([dof_event[f'kpGain{i:d}'] for i in range(22)])
            ki_array = np.array([dof_event[f'kiGain{i:d}'] for i in range(22)])
            kd_array = np.array([dof_event[f'kdGain{i:d}'] for i in range(22)])
            thisDict={"I": arrays[1],
                      "Clipped_I": arrays[2],
                      "Correction": arrays[3],
                      "Kp": kp_array,
                      "Ki": ki_array,
                      "Kd": kd_array}            
            pidDict[expId] = thisDict
filename = f"/home/c/cslage/u/MTAOS/times_square_reports/PID_data_{dayObs}.pkl"
with open(filename, 'wb') as f:
    pkl.dump(pidDict, f)
                              

## Example dictionary entry

In [ ]:
expId = 2026032500166
this_dict = pidDict[expId]
for key in thisDict.keys():
    print(key, thisDict[key])